## Tennis Scoring-Rule Validator

In [3]:
# Fixes valid set definition (7-0..7-4 are NOT valid)
# Allows match tiebreak (10-x) on deciding sets, not only doubles
# Aligns sets_played between home/away score tables

import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path('cleaned_data')  # <- پوشه‌ی پارکت‌ها

In [4]:
# 1. Load
event = pd.read_parquet(BASE_DIR / 'event.parquet')
hs    = pd.read_parquet(BASE_DIR / 'home_team_score.parquet')
as_   = pd.read_parquet(BASE_DIR / 'away_team_score.parquet')
tm    = pd.read_parquet(BASE_DIR / 'time.parquet')
tour  = pd.read_parquet(BASE_DIR / 'tournament.parquet')
home  = pd.read_parquet(BASE_DIR / 'home_team.parquet')
away  = pd.read_parquet(BASE_DIR / 'away_team.parquet')

In [5]:
# 2. Align sets_played (both sides identical)

def n_periods(df):
    return (
        df['period_1'].notna().astype('Int64')
        + df['period_2'].notna().astype('Int64')
        + df['period_3'].notna().astype('Int64')
    ).replace(0, pd.NA)

hs = hs.copy()
as_ = as_.copy()
aligned = pd.DataFrame({
    'h': n_periods(hs),
    'a': n_periods(as_),
})
aligned['sets'] = aligned[['h', 'a']].max(axis=1)
hs['sets_played'] = aligned['sets'].values
as_['sets_played'] = aligned['sets'].values

assert (hs['sets_played'].fillna(-1) == as_['sets_played'].fillna(-1)).all()

In [6]:
# 3. Merge analysis frame 
m = (
    event
    .merge(hs, on='match_id', suffixes=('', '_drop'))
    .merge(as_, on='match_id', suffixes=('_home', '_away'))
    .merge(
        tm[['match_id', 'duration_seconds', 'duration_minutes', 'duration_reliable']],
        on='match_id', how='left'
    )
    .merge(
        tour[['match_id', 'tournament_category_name', 'tournament_name']],
        on='match_id', how='left'
    )
)

In [7]:
# column names after merge: period_1_home / period_1_away from suffixes
# Safer explicit rename path:
sc = hs.merge(as_, on='match_id', suffixes=('_home', '_away'))
m = (
    event
    .merge(sc, on='match_id', how='left')
    .merge(
        tm[['match_id', 'duration_seconds', 'duration_minutes', 'duration_reliable']],
        on='match_id', how='left'
    )
    .merge(
        tour[['match_id', 'tournament_category_name', 'tournament_name']],
        on='match_id', how='left'
    )
)

In [8]:
# 4. Grain: individual match vs rows without player entities
has_home_player = m['match_id'].isin(set(home['match_id']))
has_away_player = m['match_id'].isin(set(away['match_id']))
no_periods = (
    m['period_1_home'].isna() & m['period_2_home'].isna() & m['period_3_home'].isna()
    & m['period_1_away'].isna() & m['period_2_away'].isna() & m['period_3_away'].isna()
)
is_tie_level = no_periods & ~has_home_player & ~has_away_player

im = m.loc[~is_tie_level].copy().reset_index(drop=True)
print(f"Total: {len(m)} | excluded grain (no players & no periods): {is_tie_level.sum()} | "
      f"analyzed: {len(im)}")

Total: 16873 | excluded grain (no players & no periods): 561 | analyzed: 16312


In [9]:
# 5. Set classification (CORRECT tennis rules)

def classify_set(h, a, is_last_set, allow_super_tb):
    if pd.isna(h) and pd.isna(a):
        return 'not_played'
    if pd.isna(h) or pd.isna(a):
        return 'anomaly_one_sided_null'

    h, a = int(h), int(a)
    hi, lo = max(h, a), min(h, a)

    # Standard completed set
    if (hi == 6 and lo <= 4) or (hi == 7 and lo == 5):
        return 'valid_normal'
    # Set tiebreak
    if hi == 7 and lo == 6:
        return 'valid_tiebreak'
    # Match / super tiebreak (usually deciding set)
    if allow_super_tb and hi >= 10 and (hi - lo) >= 2:
        return 'valid_super_tb'
    # Advantage set without TB (rare but real in some formats): 8-6, 9-7, ...
    if hi >= 8 and hi < 10 and (hi - lo) >= 2:
        return 'valid_advantage'
    # Impossible under standard rules (e.g. 7-3)
    if hi == 7 and lo <= 4:
        return 'anom_7'

    if is_last_set:
        return 'incomplete_last_set'   # retirement / default pattern
    return 'genuine_anomaly'

In [10]:
set_rows = []
for i, row in im.iterrows():
    played = [s for s in (1, 2, 3)
              if pd.notna(row[f'period_{s}_home']) or pd.notna(row[f'period_{s}_away'])]
    last_played = max(played) if played else 0

    for s in (1, 2, 3):
        # super TB allowed on the last played set (decider), any event type
        allow_super = (s == last_played and s >= 2)
        status = classify_set(
            row[f'period_{s}_home'],
            row[f'period_{s}_away'],
            s == last_played,
            allow_super,
        )
        set_rows.append((row['match_id'], s, status))

sets_df = pd.DataFrame(set_rows, columns=['match_id', 'set_no', 'status'])
print("\nSet-level status counts:")
print(sets_df['status'].value_counts())


Set-level status counts:
status
valid_normal              25943
not_played                15946
valid_tiebreak             2926
anom_7                     1721
valid_super_tb             1214
incomplete_last_set         681
valid_advantage             257
genuine_anomaly             236
anomaly_one_sided_null       12
Name: count, dtype: int64


In [11]:
BAD_SET = {'genuine_anomaly', 'anom_7', 'anomaly_one_sided_null'}
genuine_bad = sets_df[sets_df['status'].isin(BAD_SET)]
print(f"Matches with at least one bad set status: {genuine_bad['match_id'].nunique()}")

Matches with at least one bad set status: 1490


In [12]:
# 6. Tiebreak field consistency
tb_issues = []
for _, row in im.iterrows():
    for s in (1, 2, 3):
        h, a = row[f'period_{s}_home'], row[f'period_{s}_away']
        if pd.isna(h) or pd.isna(a):
            continue
        h, a = int(h), int(a)
        tbh = row.get(f'period_{s}_tie_break_home', np.nan)
        tba = row.get(f'period_{s}_tie_break_away', np.nan)
        # column names in merged frame
        tbh = row[f'period_{s}_tie_break_home'] if f'period_{s}_tie_break_home' in row.index else np.nan
        tba = row[f'period_{s}_tie_break_away'] if f'period_{s}_tie_break_away' in row.index else np.nan

        was_tb = (max(h, a) == 7 and min(h, a) == 6)
        tb_present = not (pd.isna(tbh) and pd.isna(tba))
        if was_tb and not tb_present:
            tb_issues.append((row['match_id'], s, 'tiebreak_set_missing_tb_score'))
        elif (not was_tb) and tb_present and max(h, a) <= 7:
            tb_issues.append((row['match_id'], s, 'tb_score_present_but_not_a_tiebreak_set'))

tb_issues_df = pd.DataFrame(tb_issues, columns=['match_id', 'set_no', 'issue'])
print(f"\nTiebreak-field inconsistencies: {len(tb_issues_df)}")


Tiebreak-field inconsistencies: 564


In [13]:
# 7. Winner vs games (diagnostic; retirements expected)

def naive_sets_won(row, side):
    other = 'away' if side == 'home' else 'home'
    n = 0
    for s in (1, 2, 3):
        hs_, as_ = row[f'period_{s}_home'], row[f'period_{s}_away']
        if pd.isna(hs_) or pd.isna(as_):
            continue
        if side == 'home' and hs_ > as_:
            n += 1
        if side == 'away' and as_ > hs_:
            n += 1
    return n

In [14]:
im['naive_sets_home'] = im.apply(lambda r: naive_sets_won(r, 'home'), axis=1)
im['naive_sets_away'] = im.apply(lambda r: naive_sets_won(r, 'away'), axis=1)
im['winner_sets'] = np.where(
    im['winner_code'] == 1, im['naive_sets_home'],
    np.where(im['winner_code'] == 2, im['naive_sets_away'], np.nan)
)

decided = im[(im['naive_sets_home'] + im['naive_sets_away']) > 0]
naive_winner = np.where(
    decided['naive_sets_home'] > decided['naive_sets_away'], 1.0,
    np.where(decided['naive_sets_away'] > decided['naive_sets_home'], 2.0, np.nan)
)
winner_contradiction = decided[
    decided['winner_code'].notna() & (decided['winner_code'] != naive_winner)
]
print(f"\nwinner_code vs games contradiction: {len(winner_contradiction)} "
      f"(many are legitimate retirements)")


winner_code vs games contradiction: 64 (many are legitimate retirements)


In [15]:
# 8. Best-of-3 overrun (only count clearly decided standard sets)
def set_winner(h, a):
    """Return 1 home, 2 away, or None if not a valid completed set."""
    if pd.isna(h) or pd.isna(a):
        return None
    h, a = int(h), int(a)
    hi, lo = max(h, a), min(h, a)
    ok = (
        (hi == 6 and lo <= 4) or (hi == 7 and lo in (5, 6))
        or (hi >= 8 and hi - lo >= 2)
    )
    if not ok:
        return None
    return 1 if h > a else 2

bo3_ids = []
for _, row in im.iterrows():
    w1 = set_winner(row['period_1_home'], row['period_1_away'])
    w2 = set_winner(row['period_2_home'], row['period_2_away'])
    if w1 is not None and w2 is not None and w1 == w2:
        if pd.notna(row['period_3_home']) or pd.notna(row['period_3_away']):
            bo3_ids.append(row['match_id'])
print(f"Best-of-3 overrun (2-0 but 3rd set present): {len(bo3_ids)}")

Best-of-3 overrun (2-0 but 3rd set present): 0


In [16]:
# 9. Duration
dur = im[im['duration_seconds'].notna() & im['duration_minutes'].notna()].copy()
dur['diff'] = (dur['duration_seconds'] - dur['duration_minutes'] * 60).abs()
implausible = dur[(dur['duration_minutes'] < 15) | (dur['duration_minutes'] > 300)]
print(f"duration sec/min mismatch >30s: {(dur['diff'] > 30).sum()}")
print(f"Implausible duration (<15 or >300 min): {len(implausible)}")

duration sec/min mismatch >30s: 0
Implausible duration (<15 or >300 min): 17


In [17]:
# 10. Walkovers / completed with no set data
no_per = (
    im['period_1_home'].isna() & im['period_2_home'].isna() & im['period_3_home'].isna()
    & im['period_1_away'].isna() & im['period_2_away'].isna() & im['period_3_away'].isna()
)
completed_no_data = im[im['match_completed'] & no_per]
print(f"completed with no set data (walkovers?): {len(completed_no_data)}")

completed with no set data (walkovers?): 51


In [18]:
# 11. Per-match quality flags
bad_match_ids = set(genuine_bad['match_id'])
flags = pd.DataFrame({'match_id': im['match_id'].values})
flags['is_tie_level'] = False
flags['has_genuine_set_anomaly'] = flags['match_id'].isin(bad_match_ids)
flags['has_tiebreak_field_issue'] = flags['match_id'].isin(set(tb_issues_df['match_id']))
flags['winner_code_contradicts_games'] = flags['match_id'].isin(set(winner_contradiction['match_id']))
flags['bo3_overrun'] = flags['match_id'].isin(set(bo3_ids))
flags['implausible_duration'] = flags['match_id'].isin(set(implausible['match_id']))
flags['completed_with_no_set_data'] = flags['match_id'].isin(set(completed_no_data['match_id']))
flags['is_retirement_like'] = flags['match_id'].isin(
    set(im.loc[(im['match_completed']) & (im['winner_code'].notna()) & (im['winner_sets'] < 2), 'match_id'])
)

In [19]:
# set_scores_valid: no bad set statuses
flags['set_scores_valid'] = ~flags['has_genuine_set_anomaly']

# score_ready: completed + winner + sets_played present
sp = hs.set_index('match_id')['sets_played']
flags = flags.merge(im[['match_id', 'match_completed', 'winner_code']], on='match_id', how='left')
flags['sets_played'] = flags['match_id'].map(sp)
flags['score_ready'] = (
    (flags['match_completed'] == True)
    & flags['winner_code'].notna()
    & flags['sets_played'].notna()
)
flags = flags.drop(columns=['match_completed', 'winner_code', 'sets_played'])

In [20]:
# add excluded grain rows
tie_flags = pd.DataFrame({'match_id': m.loc[is_tie_level, 'match_id'].values})
tie_flags['is_tie_level'] = True
for c in flags.columns:
    if c not in tie_flags.columns:
        tie_flags[c] = False

quality_flags = pd.concat([flags, tie_flags], ignore_index=True)

In [21]:
# 12. Write outputs

# score tables with aligned sets_played
hs.to_parquet(BASE_DIR / 'home_team_score.parquet', index=False)
as_.to_parquet(BASE_DIR / 'away_team_score.parquet', index=False)

In [22]:
# event with consolidated analysis flags
ev = event.drop(
    columns=[c for c in ('score_ready','set_scores_valid', 'is_retirement_like') if c in event.columns],
    errors='ignore'
)
ev = ev.merge(
    quality_flags[['match_id', 'score_ready','set_scores_valid', 'is_retirement_like']],
    on='match_id',
    how='left'
)
for c in ( 'score_ready','set_scores_valid', 'is_retirement_like'):
    ev[c] = ev[c].fillna(False).astype(bool)

ev.to_parquet(BASE_DIR / 'event.parquet', index=False)
quality_flags.to_csv(BASE_DIR / 'tennis_data_quality_flags.csv', index=False)

In [23]:
print("\n=== SAVED ===")
print("  home_team_score.parquet  (sets_played aligned)")
print("  away_team_score.parquet  (sets_played aligned)")
print("  event.parquet            (+ set_scores_valid, is_retirement_like)")
print("  tennis_data_quality_flags.csv")
print(f"\nscore_ready:        {ev['score_ready'].sum()}")
print(f"set_scores_valid:   {ev['set_scores_valid'].sum()}")
print(f"is_retirement_like: {ev['is_retirement_like'].sum()}")


=== SAVED ===
  home_team_score.parquet  (sets_played aligned)
  away_team_score.parquet  (sets_played aligned)
  event.parquet            (+ set_scores_valid, is_retirement_like)
  tennis_data_quality_flags.csv

score_ready:        14413
set_scores_valid:   14822
is_retirement_like: 275
